# 01. Historical extraction: Banco de Portugal / INE

Extract the 1977-1995 balance, account, transfer and macro tables directly from the bundled official long-series workbook, and check that the extracted accounts satisfy revenue minus expenditure equals the balance.

**Reads**

- `data/raw/banco_portugal/series_longas_2023-12.xlsx`

**Writes**

- `data/interim/historical_balances_1977_1995.csv`
- `data/interim/historical_accounts_1977_1995.csv`
- `data/interim/historical_intragov_transfers_1977_1995.csv`
- `data/interim/historical_macro_1977_1995.csv`

**Method reference:** `METHODOLOGY.md` sections 3-4

In [ ]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

%matplotlib inline

from portugal_fiscal_balance.analysis import figures

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

## 1. Parse the workbook

The historical workbook holds sector accounts, B.9 balances, intragovernmental
transfers, GDP and labour-market series across several sheets and layouts. The
parser lives in `portugal_fiscal_balance.sources.banco_portugal`, so the parsing
rules are testable and this notebook stays a narrative.

In [ ]:
from portugal_fiscal_balance.io import write_csv
from portugal_fiscal_balance.sources.banco_portugal import extract_long_series

historical = extract_long_series(RAW / 'banco_portugal' / 'series_longas_2023-12.xlsx')
shapes = pd.DataFrame(
    [
        {'table': 'balances', 'rows': historical.balances.shape[0], 'columns': historical.balances.shape[1]},
        {'table': 'accounts', 'rows': historical.accounts.shape[0], 'columns': historical.accounts.shape[1]},
        {'table': 'transfers', 'rows': historical.transfers.shape[0], 'columns': historical.transfers.shape[1]},
        {'table': 'macro', 'rows': historical.macro.shape[0], 'columns': historical.macro.shape[1]},
    ]
)
display(shapes)

## 2. The historical balance panel

The four balances are reported in millions of euro alongside nominal GDP, which
makes the GDP ratios reproducible rather than copied from a published table.

In [ ]:
balance_columns = [
    'year',
    'general_government_balance_m_eur',
    'central_government_balance_m_eur',
    'regional_local_balance_m_eur',
    'social_security_balance_m_eur',
    'nominal_gdp_m_eur',
]
display(historical.balances[balance_columns].tail(8).round(1))

In [ ]:
figure = figures.balances_by_subsector(
    historical.balances,
    title='Historical long series: fiscal balance by subsector, 1977-1995',
    splice_year=None,
)

The 1995 observation is extracted here but is **not** used in the canonical
panel: notebook 03 takes 1995 from the modern source and keeps this vintage only
as an overlap diagnostic.

## 3. Detailed subsector accounts

In [ ]:
account_columns = [
    'year',
    'sector',
    'total_revenue_m_eur',
    'total_expenditure_m_eur',
    'interest_m_eur',
    'gfcf_m_eur',
    'balance_m_eur',
]
recent_accounts = historical.accounts.loc[historical.accounts['year'].ge(1993), account_columns]
display(recent_accounts.sort_values(['sector', 'year']).round(1))

## 4. Extraction check

If the parser mapped a row incorrectly, revenue minus expenditure would stop
reproducing the published balance. The residual below is therefore an extraction
test, not an economic result.

In [ ]:
identity = (
    historical.accounts.groupby('sector')['account_identity_error_m_eur']
    .apply(lambda column: column.abs().max())
    .rename('max_abs_identity_error_m_eur')
    .to_frame()
)
display(identity)
print('worst absolute account identity error (M EUR):', float(identity.max().iloc[0]))

## 5. Transfers and macro context

The transfer table supports the mechanical sensitivity in notebook 10. The macro
table supplies the historical labour-market controls used in notebook 14.

In [ ]:
display(historical.transfers.loc[historical.transfers['year'].ge(1993)].round(1))
display(historical.macro.tail(6).round(3))

## 6. Persist the extraction

Extracted tables are written to `data/interim` under their source name. Keeping
extraction output separate from processed panels makes it possible to attribute a
later disagreement to a specific source.

In [ ]:
write_csv(historical.balances, INTERIM / 'historical_balances_1977_1995.csv')
write_csv(historical.accounts, INTERIM / 'historical_accounts_1977_1995.csv')
write_csv(historical.transfers, INTERIM / 'historical_intragov_transfers_1977_1995.csv')
write_csv(historical.macro, INTERIM / 'historical_macro_1977_1995.csv')
print('written to', INTERIM.relative_to(ROOT))

## Interpretation limits

1. These are **historical-vintage** national accounts. They are not directly
   comparable with post-1995 ESA 2010 figures, and this repository never
   calibrates one to the other.
2. The workbook is a **secondary compilation** of historical statistics. Its own
   revisions are outside the scope of this repository.
3. Labour-market series from this period are used only as descriptive controls in
   notebook 14.

---

[Previous: 00. Research protocol](00_research_protocol.ipynb) | [Next: 02. Modern extraction: INE/PORDATA and CFP](02_extract_modern_data.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```